## Serialize Tabular Data using SchemaOrg and the CUAHSI.org-ScientificDataset extension

The purpose of this notebook is to evaluate how Raster data can be extracted and mapped to our Pydantic classes. It demonstrates how three common formats can be encoded in the CUAHSI.org ScientificDataset class: `CSV`, `Parquet`.


Each of these data formats is mapped to the ScientificDataset class via the following relationships:


|ScientificDataset Attribute|What it Describes|Example|
|---|---|---|
|Dimension	|The number of feature in the dataset	|rows = 12, columns=10, bands=5 |
|Variable	|The data variable represented by the raster grid | elevation(band, row, col) |


In [1]:
import os
import sys
import pandas
import hashlib
import rasterio
import mimetypes
from glob import glob
from pyproj import CRS
from pathlib import Path

# add the parent directory to the path. This is the 
# directory that contains our pydantic classes.
sys.path.append('..')
import base
import core
import dataset
import datavariable

In [2]:
def compute_sha256(file_path: Path) -> str:
    """Computes the SHA256 hash of a file.

    Args:
        file_path: The path to the file.

    Returns:
        The hexadecimal representation of the SHA256 hash.
    """
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            sha256_hash.update(chunk)
    return sha256_hash.hexdigest()

In [3]:
# Add raster MIME types if not already present
mimetypes.add_type("text/csv", ".csv")
mimetypes.add_type("application/vnd.apache.parquet", ".parquet")

In [95]:
def read_generic(filepath, delimiter=',', comment='#', skiprows=[], header=0, parse_all_dates=False):
    df = pandas.read_csv(filepath, sep=delimiter, comment=comment, skiprows=skiprows, header=header)

    # convert date columns into datetime64
    if parse_all_dates:
        for col in df.columns:
            if ('date' in col.lower()) or ('time' in col.lower()):
                try:
                    converted = pandas.to_datetime(df[col], errors='raise')
                
                    # Check if a reasonable number of non-NaT values exist after conversion
                    if converted.notna().sum() > 0:
                        df[col] = converted
                    
                except (ValueError, TypeError):
                    # Not a date column
                    pass
    return df

def read_nwis(filepath):
    df = read_generic(filepath,
                      delimiter='\t',
                      comment='#',
                      header=0)

    # remove the column widths row
    df = df.iloc[1:].reset_index(drop=True)
    
    # create a tz-aware datetime index
    df["datetime"] = pandas.to_datetime(df['datetime'], format="%Y-%m-%d %H:%M")
    df.set_index("datetime", inplace=True)

    return df
    

def encode_pandas_dataframe(df, filepath):

    # Build dataset dimensions.
    dimensions = []
    coordinates = []
    idx_name = 'index' if df.index.name is None else df.index.name
    print(f'INDEX NAME: {idx_name}')
    
    # build dimension from index
    dimensions.append(
        datavariable.Dimension(
            name =  idx_name,
            shape = len(df.index)
        )
    )
    # build coordinate from index
    coordinates.append(
        datavariable.DataVariable(
            name = idx_name,
            dataType = str(df.index.dtype),
            minValue = str(df.index.min()),
            maxValue = str(df.index.max()),
            dimensions=idx_name,
        )
    )           

    # define a list of variable names for which we will not compute statistics. This
    # is because min, max, nodata don't make much sense for some of these.
    skip_stats_for_variables = ['geometry', 'geom'] 
    
    variables = []
    for col_name in df.columns:
        minValue = None
        maxValue = None
        if col_name not in skip_stats_for_variables:            
            minValue = str(df[col_name].min())
            maxValue = str(df[col_name].max())
        variables.append(
            datavariable.DataVariable(
                name =col_name,
                dimensions=dimensions[0].name,
                dataType = str(df[col_name].dtype),
                minValue = minValue,
                maxValue = maxValue,
                
            )
        )
    ######### 
    # TODO: # Modify this to support remote s3 files as well as local
    #########
    
    # files = [
    #         base.MediaObject(contentUrl = filepath,
    #                          name = filepath,
    #                          #sha256 = sha256,
    #                          #contentSize = contentSize,
    #                          #encodingFormat = encodingFormat
    #                         )
    #     ]
    
    # get all file names that match the patter of the input filepath
    search_path = f"{'.'.join(filepath.split('.')[:-1])}.*"
    associated_files = glob(search_path)
    files = []
    for fpath in associated_files:
        files.append(
            base.MediaObject(
                contentUrl = f'https://hydroshare.org/my-resource/{fpath}',
                name = Path(fpath).name,
                sha256 = compute_sha256(Path(fpath)),
                contentSize = f'{os.path.getsize(Path(fpath))/1024} KB',
                encodingFormat = mimetypes.guess_type(Path(fpath))[0],
            )
        )

    return dataset.ScientificDataset(
        variableMeasured = variables,
        dimensions = dimensions,
        coordinates = coordinates,
        associatedMedia=files,
        additionalType=dataset.AdditionalType.TABULAR,
    )

In [73]:
# def encode_delimited_metadata(filepath, delimiter=',', dimension_column_names=None):
#     df = pandas.read_csv(filepath, comment='#')

#     # convert date columns into datetime64
#     for col in df.columns:
#         if ('date' in col.lower()) or ('time' in col.lower()):
#             try:
#                 converted = pandas.to_datetime(df[col], errors='raise')
            
#                 # Check if a reasonable number of non-NaT values exist after conversion
#                 if converted.notna().sum() > 0:
#                     df[col] = converted
                
#             except (ValueError, TypeError):
#                 # Not a date column
#                 pass

#     # Build dataset dimensions.
#     dimensions = []
#     coordinates = []
#     if dimension_column_names is None:
#         # build dimension from index
#         dimensions.append(
#             datavariable.Dimension(
#                 name = 'index',
#                 shape = len(df.index)
#             )
#         )
#         # build coordinate from index
#         coordinates.append(
#             datavariable.DataVariable(
#                 name = 'index',
#                 dataType = str(df.index.dtype),
#                 minValue = str(df.index.min()),
#                 maxValue = str(df.index.max()),
#                 dimensions='index',
#             )
#         )

        
#     else:
#         for col_name in dimension_column_names:
#             dimensions.append(
#                 datavariable.Dimension(
#                     name = col_name,
#                     shape = len(df[col_name])
#                 ))            
            
#     variables = []
#     for col_name in df.columns:
#         variables.append(
#             datavariable.DataVariable(
#                 name =col_name,
#                 dataType = str(df[col_name].dtype),
#                 minValue = str(df[col_name].min()),
#                 maxValue = str(df[col_name].max()),
#                 dimensions=dimensions[0].name,
#             )
#         )


#     # get all file names that match the patter of the input filepath
#     search_path = f"{'.'.join(filepath.split('.')[:-1])}.*"
#     associated_files = glob(search_path)
#     files = []
#     for fpath in associated_files:
#         files.append(
#             base.MediaObject(
#                 contentUrl = f'https://hydroshare.org/my-resource/{fpath}',
#                 name = Path(fpath).name,
#                 sha256 = compute_sha256(Path(fpath)),
#                 contentSize = f'{os.path.getsize(Path(fpath))/1024} KB',
#                 encodingFormat = mimetypes.guess_type(Path(fpath))[0],
#             )
#         )

#     return dataset.ScientificDataset(
#         variableMeasured = variables,
#         dimensions = dimensions,
#         coordinates = coordinates,
#         associatedMedia=files,
#         additionalType=dataset.AdditionalType.TABULAR,
#     )


### Encode a CSV File with Unknown Dimensions

In [72]:
df = read_generic('./data/LR_GC_C_SourceID_1_QC_0_Year_2014.csv')
meta = encode_pandas_dataframe(df, './data/LR_GC_C_SourceID_1_QC_0_Year_2014.csv')
print(meta.model_dump_json(exclude_none=True, indent=4))

INDEX NAME: index
{
    "context": "https://hydroshare.org/schema",
    "type": "ScientificDataset",
    "associatedMedia": [
        {
            "type": "MediaObject",
            "contentUrl": "https://hydroshare.org/my-resource/data/LR_GC_C_SourceID_1_QC_0_Year_2014.csv",
            "encodingFormat": "text/csv",
            "contentSize": "16169.5390625 KB",
            "name": "LR_GC_C_SourceID_1_QC_0_Year_2014.csv",
            "sha256": "f6db340e040de6a9c513af04e11f8bdad3f476dc1c1ee1c0064918974ad1b437"
        }
    ],
    "variableMeasured": [
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "LocalDateTime",
            "dimensions": "index",
            "dataType": "object",
            "minValue": "2014-01-14 10:00:00",
            "maxValue": "2014-12-31 23:45:00"
        },
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "UTCOf

### Encode a USGS NWIS Tab Separated File

In [75]:
df = read_nwis('./data/usgs_nwis_gills_creek.txt')
meta = encode_pandas_dataframe(df, './data/usgs_nwis_gills_creek.txt')
print(meta.model_dump_json(exclude_none=True, indent=4))

INDEX NAME: datetime
{
    "context": "https://hydroshare.org/schema",
    "type": "ScientificDataset",
    "associatedMedia": [
        {
            "type": "MediaObject",
            "contentUrl": "https://hydroshare.org/my-resource/data/usgs_nwis_gills_creek.txt",
            "encodingFormat": "text/plain",
            "contentSize": "28.771484375 KB",
            "name": "usgs_nwis_gills_creek.txt",
            "sha256": "9e3a7bf4a3969932ae8a0fceac7c19b39105a115eaa91e7900d55629a193e1df"
        }
    ],
    "variableMeasured": [
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "agency_cd",
            "dimensions": "datetime",
            "dataType": "object",
            "minValue": "USGS",
            "maxValue": "USGS"
        },
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "site_no",
            "dimensions": "datetime",
        

### Encode Parquet

In [82]:
!pip install s3fs pyarrow -q


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [83]:
import s3fs 

In [97]:
url = 's3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/MA_Wetlands.parquet'
df = pandas.read_parquet(url, engine="pyarrow")

In [96]:
meta = encode_pandas_dataframe(df, 's3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/MA_Wetlands.parquet')
print(meta.model_dump_json(exclude_none=True, indent=4))

INDEX NAME: index
{
    "context": "https://hydroshare.org/schema",
    "type": "ScientificDataset",
    "associatedMedia": [],
    "variableMeasured": [
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "ATTRIBUTE",
            "dimensions": "index",
            "dataType": "object",
            "minValue": "E1AB3L",
            "maxValue": "R5USC"
        },
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "WETLAND_TYPE",
            "dimensions": "index",
            "dataType": "object",
            "minValue": "Estuarine and Marine Deepwater",
            "maxValue": "Riverine"
        },
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "ACRES",
            "dimensions": "index",
            "dataType": "float64",
            "minValue": "1.237145693853e-07",
       